In [0]:
%sql
-- KPI 1: Month-to-Date Performance Comparison
-- Compares current MTD with previous month's MTD for same day
-- Uses LAG window function for period-over-period analysis

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_01_mtd_performance AS
WITH daily_orders AS (
  SELECT
    store_id,
    store_name,
    order_year,
    order_month,
    DAY(order_date) as day_of_month,
    SUM(total_invoice_amount) as daily_revenue,
    COUNT(CASE WHEN order_status = 'COMPLETED' THEN order_id END) as daily_completed_orders
  FROM mini_project_catalog.03_gold.fact_orders
  GROUP BY store_id, store_name, order_year, order_month, DAY(order_date)
),
mtd_cumulative AS (
  SELECT
    store_id,
    store_name,
    order_year,
    order_month,
    day_of_month,
    SUM(daily_revenue) OVER (
      PARTITION BY store_id, order_year, order_month 
      ORDER BY day_of_month
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as mtd_revenue,
    SUM(daily_completed_orders) OVER (
      PARTITION BY store_id, order_year, order_month 
      ORDER BY day_of_month
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) as mtd_completed_orders
  FROM daily_orders
)
SELECT
  store_id,
  store_name,
  order_year,
  order_month,
  day_of_month,
  mtd_revenue,
  mtd_completed_orders,
  LAG(mtd_revenue) OVER (
    PARTITION BY store_id, day_of_month 
    ORDER BY order_year, order_month
  ) as prev_month_mtd_revenue,
  LAG(mtd_completed_orders) OVER (
    PARTITION BY store_id, day_of_month 
    ORDER BY order_year, order_month
  ) as prev_month_mtd_orders,
  ROUND(
    ((mtd_revenue - LAG(mtd_revenue) OVER (
      PARTITION BY store_id, day_of_month 
      ORDER BY order_year, order_month
    )) / NULLIF(LAG(mtd_revenue) OVER (
      PARTITION BY store_id, day_of_month 
      ORDER BY order_year, order_month
    ), 0)) * 100, 2
  ) as revenue_growth_pct,
  ROUND(
    ((mtd_completed_orders - LAG(mtd_completed_orders) OVER (
      PARTITION BY store_id, day_of_month 
      ORDER BY order_year, order_month
    )) / NULLIF(LAG(mtd_completed_orders) OVER (
      PARTITION BY store_id, day_of_month 
      ORDER BY order_year, order_month
    ), 0)) * 100, 2
  ) as orders_growth_pct
FROM mtd_cumulative
ORDER BY store_id, order_year, order_month, day_of_month;

SELECT CONCAT('✅ KPI 1: MTD Performance created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_01_mtd_performance;
SELECT * FROM mini_project_catalog.03_gold.kpi_01_mtd_performance

In [0]:
%sql
-- KPI 2: Average Days in Shop by Store and Service Type
-- Shows operational efficiency metrics

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_02_days_in_shop AS
SELECT
  store_id,
  store_name,
  city,
  state,
  service_type,
  ROUND(AVG(days_in_shop), 2) as avg_days_in_shop,
  COUNT(order_id) as total_orders
FROM mini_project_catalog.03_gold.fact_orders
WHERE order_status IN ('COMPLETED', 'IN_PROGRESS')
  AND days_in_shop IS NOT NULL
GROUP BY store_id, store_name, city, state, service_type
ORDER BY store_id, service_type;

SELECT CONCAT('✅ KPI 2: Average Days in Shop created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_02_days_in_shop;
SELECT * FROM mini_project_catalog.03_gold.kpi_02_days_in_shop

In [0]:
%sql
-- KPI 3: Survey Coverage (Response Rates by Store)
-- Tracks customer engagement metrics

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_03_survey_coverage AS
SELECT
  store_id,
  COUNT(*) as surveys_sent,
  COUNT(CASE WHEN responded_flag = TRUE THEN survey_id END) as surveys_responded,
  ROUND(
    COUNT(CASE WHEN responded_flag = TRUE THEN survey_id END) * 100.0 / COUNT(*), 2
  ) as response_rate_pct
FROM mini_project_catalog.03_gold.fact_surveys
GROUP BY store_id
ORDER BY response_rate_pct DESC;

SELECT CONCAT('✅ KPI 3: Survey Coverage created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_03_survey_coverage;
SELECT * FROM mini_project_catalog.03_gold.kpi_03_survey_coverage

In [0]:
%sql
-- KPI 4: Overall Satisfaction Scores with Store Ranking
-- Aggregates survey ratings by store

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_04_survey_scores AS
WITH store_scores AS (
  SELECT
    f.store_id,
    f.store_name,
    ROUND(AVG(s.overall_satisfaction_rating), 2) as avg_overall_satisfaction,
    ROUND(AVG(s.work_quality_rating), 2) as avg_work_quality,
    ROUND(AVG(s.communication_rating), 2) as avg_communication,
    ROUND(AVG(s.cleanliness_rating), 2) as avg_cleanliness,
    COUNT(s.survey_id) as total_surveys
  FROM mini_project_catalog.03_gold.fact_surveys s
  JOIN mini_project_catalog.03_gold.fact_orders f ON s.order_id = f.order_id
  WHERE s.responded_flag = TRUE
  GROUP BY f.store_id, f.store_name
)
SELECT
  *,
  RANK() OVER (ORDER BY avg_overall_satisfaction DESC) as satisfaction_rank
FROM store_scores
ORDER BY satisfaction_rank;

SELECT CONCAT('✅ KPI 4: Survey Scores Summary created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_04_survey_scores;
SELECT * FROM mini_project_catalog.03_gold.kpi_04_survey_scores

In [0]:
%sql
-- KPI 5: Revenue vs Budget by Manager (Monthly)
-- Shows budget achievement with ranking

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_05_revenue_vs_budget AS
WITH manager_revenue AS (
  SELECT
    st.manager_id,
    st.manager_name,
    f.order_year as year,
    f.order_month as month,
    ROUND(SUM(f.total_invoice_amount), 2) as actual_revenue
  FROM mini_project_catalog.03_gold.fact_orders f
  JOIN mini_project_catalog.02_silver.store st ON f.store_id = st.store_id
  WHERE f.order_status = 'COMPLETED'
  GROUP BY st.manager_id, st.manager_name, f.order_year, f.order_month
),
manager_budget AS (
  SELECT
    st.manager_id,
    YEAR(b.month) as year,
    MONTH(b.month) as month,
    SUM(b.budget_amount) as budget_amount
  FROM mini_project_catalog.02_silver.ns_budget b
  JOIN mini_project_catalog.02_silver.store st ON b.ns_store_id = st.store_id
  GROUP BY st.manager_id, YEAR(b.month), MONTH(b.month)
),
revenue_budget_compare AS (
  SELECT
    mr.manager_id,
    mr.manager_name,
    mr.year,
    mr.month,
    mr.actual_revenue,
    COALESCE(mb.budget_amount, 0) as budget_amount,
    ROUND((mr.actual_revenue / NULLIF(mb.budget_amount, 0)) * 100, 2) as budget_achievement_pct,
    ROUND(mr.actual_revenue - COALESCE(mb.budget_amount, 0), 2) as variance_amount
  FROM manager_revenue mr
  LEFT JOIN manager_budget mb 
    ON mr.manager_id = mb.manager_id 
    AND mr.year = mb.year 
    AND mr.month = mb.month
)
SELECT
  *,
  RANK() OVER (
    PARTITION BY year, month 
    ORDER BY budget_achievement_pct DESC
  ) as achievement_rank
FROM revenue_budget_compare
ORDER BY year, month, achievement_rank;

SELECT CONCAT('✅ KPI 5: Revenue vs Budget created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_05_revenue_vs_budget;
SELECT * FROM mini_project_catalog.03_gold.kpi_05_revenue_vs_budget

In [0]:
%sql
-- KPI 6: Top 5 Technicians by Completion Time Accuracy (per Store-Month)
-- Uses ROW_NUMBER() to get exactly 5 technicians per store-month
-- Fixed: Changed from RANK() to ROW_NUMBER() to handle ties properly

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_06_top_technicians AS
WITH technician_performance AS (
  SELECT
    store_id,
    store_name,
    technician_id,
    technician_name,
    order_year,
    order_month,
    COUNT(order_id) as total_orders,
    SUM(CASE WHEN completed_on_time = 1 THEN 1 ELSE 0 END) as on_time_completions,
    ROUND((SUM(CASE WHEN completed_on_time = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(order_id)), 2) as completion_accuracy_pct
  FROM mini_project_catalog.03_gold.fact_orders
  WHERE order_status = 'COMPLETED'
    AND technician_id IS NOT NULL
  GROUP BY store_id, store_name, technician_id, technician_name, order_year, order_month
),
ranked_technicians AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY store_id, order_year, order_month 
      ORDER BY completion_accuracy_pct DESC, total_orders DESC, technician_id
    ) as accuracy_rank
  FROM technician_performance
)
SELECT
  store_id,
  store_name,
  technician_id,
  technician_name,
  order_year,
  order_month,
  total_orders,
  on_time_completions,
  completion_accuracy_pct,
  accuracy_rank
FROM ranked_technicians
WHERE accuracy_rank <= 5
ORDER BY store_id, order_year, order_month, accuracy_rank;

SELECT CONCAT('✅ KPI 6: Top Technicians created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_06_top_technicians;
SELECT * FROM mini_project_catalog.03_gold.kpi_06_top_technicians

In [0]:
%sql
-- KPI 7: Year-to-Date Revenue Growth vs Previous Year
-- Compares YTD performance year-over-year with store ranking

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_07_ytd_revenue_growth AS
WITH ytd_revenue AS (
  SELECT
    store_id,
    store_name,
    order_year,
    ROUND(SUM(total_invoice_amount), 2) as ytd_revenue
  FROM mini_project_catalog.03_gold.fact_orders
  WHERE order_status = 'COMPLETED'
  GROUP BY store_id, store_name, order_year
),
yoy_comparison AS (
  SELECT
    store_id,
    store_name,
    order_year,
    ytd_revenue,
    LAG(ytd_revenue) OVER (
      PARTITION BY store_id 
      ORDER BY order_year
    ) as prev_year_ytd_revenue,
    ROUND(
      ((ytd_revenue - LAG(ytd_revenue) OVER (
        PARTITION BY store_id 
        ORDER BY order_year
      )) / NULLIF(LAG(ytd_revenue) OVER (
        PARTITION BY store_id 
        ORDER BY order_year
      ), 0)) * 100, 2
    ) as ytd_growth_pct,
    ROUND(
      ytd_revenue - LAG(ytd_revenue) OVER (
        PARTITION BY store_id 
        ORDER BY order_year
      ), 2
    ) as ytd_growth_amount
  FROM ytd_revenue
)
SELECT
  *,
  RANK() OVER (
    PARTITION BY order_year 
    ORDER BY ytd_growth_pct DESC
  ) as growth_rank
FROM yoy_comparison
WHERE prev_year_ytd_revenue IS NOT NULL
ORDER BY order_year, growth_rank;

SELECT CONCAT('✅ KPI 7: YTD Revenue Growth created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_07_ytd_revenue_growth;
SELECT * FROM mini_project_catalog.03_gold.kpi_07_ytd_revenue_growth

In [0]:
%sql
-- KPI 8: Stage-wise Day Cycle Time Analysis
-- Breaks down total cycle time into stages by store and service type

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_08_stage_cycle_time AS
SELECT
  store_id,
  store_name,
  service_type,
  order_year,
  order_month,
  ROUND(AVG(days_vehicle_in_to_work_start), 2) as avg_days_to_start,
  ROUND(AVG(days_work_start_to_completion), 2) as avg_days_in_work,
  ROUND(AVG(days_completion_to_delivery), 2) as avg_days_to_deliver,
  ROUND(AVG(days_vehicle_in_to_work_start + days_work_start_to_completion + days_completion_to_delivery), 2) as total_cycle_time
FROM mini_project_catalog.03_gold.fact_orders
WHERE order_status = 'COMPLETED'
  AND days_vehicle_in_to_work_start IS NOT NULL
  AND days_work_start_to_completion IS NOT NULL
  AND days_completion_to_delivery IS NOT NULL
GROUP BY store_id, store_name, service_type, order_year, order_month
ORDER BY store_id, service_type, order_year, order_month;

SELECT CONCAT('✅ KPI 8: Stage Cycle Time created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_08_stage_cycle_time;
SELECT * FROM mini_project_catalog.03_gold.kpi_08_stage_cycle_time

In [0]:
%sql
-- KPI 9: Estimator Accuracy Ranking
-- Compares initial estimates vs final invoices by estimator

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_09_estimator_accuracy AS
WITH estimator_performance AS (
  SELECT
    estimator_id,
    estimator_name,
    COUNT(order_id) as total_estimates,
    ROUND(AVG(estimate_amount), 2) as avg_estimate_amount,
    ROUND(AVG(total_invoice_amount), 2) as avg_invoice_amount,
    ROUND(AVG(estimate_accuracy_pct), 2) as accuracy_pct,
    ROUND(AVG(estimate_variance), 2) as avg_variance_amount
  FROM mini_project_catalog.03_gold.fact_orders
  WHERE estimate_amount IS NOT NULL
    AND total_invoice_amount IS NOT NULL
    AND estimator_id IS NOT NULL
  GROUP BY estimator_id, estimator_name
)
SELECT
  *,
  RANK() OVER (ORDER BY accuracy_pct DESC) as accuracy_rank
FROM estimator_performance
ORDER BY accuracy_rank;

SELECT CONCAT('✅ KPI 9: Estimator Accuracy created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_09_estimator_accuracy;
SELECT * FROM mini_project_catalog.03_gold.kpi_09_estimator_accuracy

In [0]:
%sql
-- KPI 10: Technician Workload by Month
-- Tracks orders and days per technician with ranking

CREATE OR REPLACE TABLE mini_project_catalog.03_gold.kpi_10_technician_workload AS
WITH technician_workload AS (
  SELECT
    technician_id,
    technician_name,
    store_id,
    store_name,
    order_year,
    order_month,
    COUNT(order_id) as orders_assigned,
    ROUND(SUM(days_in_shop), 2) as total_days_worked,
    ROUND(AVG(days_in_shop), 2) as avg_days_per_order
  FROM mini_project_catalog.03_gold.fact_orders
  WHERE technician_id IS NOT NULL
  GROUP BY technician_id, technician_name, store_id, store_name, order_year, order_month
)
SELECT
  *,
  RANK() OVER (
    PARTITION BY store_id, order_year, order_month 
    ORDER BY orders_assigned DESC
  ) as workload_rank
FROM technician_workload
ORDER BY store_id, order_year, order_month, workload_rank;

SELECT CONCAT('✅ KPI 10: Technician Workload created with ', COUNT(*), ' records') as status
FROM mini_project_catalog.03_gold.kpi_10_technician_workload;
SELECT * FROM mini_project_catalog.03_gold.kpi_10_technician_workload

In [0]:
%sql
-- Final Verification: Count all 10 KPI tables
SELECT 'kpi_01_mtd_performance' as kpi_table, COUNT(*) as records 
FROM mini_project_catalog.03_gold.kpi_01_mtd_performance
UNION ALL 
SELECT 'kpi_02_days_in_shop', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_02_days_in_shop
UNION ALL 
SELECT 'kpi_03_survey_coverage', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_03_survey_coverage
UNION ALL 
SELECT 'kpi_04_survey_scores', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_04_survey_scores
UNION ALL 
SELECT 'kpi_05_revenue_vs_budget', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_05_revenue_vs_budget
UNION ALL 
SELECT 'kpi_06_top_technicians', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_06_top_technicians
UNION ALL 
SELECT 'kpi_07_ytd_revenue_growth', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_07_ytd_revenue_growth
UNION ALL 
SELECT 'kpi_08_stage_cycle_time', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_08_stage_cycle_time
UNION ALL 
SELECT 'kpi_09_estimator_accuracy', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_09_estimator_accuracy
UNION ALL 
SELECT 'kpi_10_technician_workload', COUNT(*) 
FROM mini_project_catalog.03_gold.kpi_10_technician_workload
ORDER BY kpi_table;